> **Public evidence copy:** Stored outputs and execution counts have been removed. 
> The participant-level workbook is not included. Set `JAI_SURVEY_DATA` to an 
> authorized secure copy before running this notebook.


# Dog-related nuisance, bites, and management

## Setup

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os 

In [ ]:
from pathlib import Path

data_path_value = os.environ.get("JAI_SURVEY_DATA")
if not data_path_value:
    raise RuntimeError(
        "Set JAI_SURVEY_DATA to the absolute path of the restricted survey workbook. "
        "Do not copy participant data into this repository."
    )

file = Path(data_path_value).expanduser().resolve()
if not file.is_file():
    raise FileNotFoundError(f"Restricted survey workbook not found: {file}")



In [ ]:
df1 = pd.read_excel(file, 'Jai KAP Survey 2025')


In [ ]:
df1_filtered = df1[
    (df1['Is someone >18 present at the household?'] == 'Yes') &
    (df1['Has the respondent consented to undertake the survey?'] == 'Yes')
]



In [ ]:
df1_ong = df1_filtered[df1_filtered["District"] == "Ongkharak"]
df1_pak = df1_filtered[df1_filtered["District"] == "Pak Phli"]


> Records without consent or without an eligible adult respondent are excluded according to the analysis cleaning rule.


## Analysis


In [ ]:
df1_filtered['In the last month, have you been annoyed or troubled by a dog or dogs in your neighbourhood?']

In [ ]:
import plotly.express as px
import pandas as pd

col = "In the last month, have you been annoyed or troubled by a dog or dogs in your neighbourhood?"

# Count responses per district
counts = (
    df1_filtered.groupby("District")[col]
    .value_counts()
    .rename_axis(["District", "Response"])
    .reset_index(name="Count")
)

# Plot
fig = px.bar(
    counts,
    x="Response",
    y="Count",
    color="District",
    barmode="group",
    text="Count",
    color_discrete_map={
        "Pak Phli": "#636efa",     # green (CNVR)
        "Ongkharak": "#ef553b"     # orange (Non-CNVR)
    },
    title="Have You Been Troubled by Dogs in the Last Month?",
    labels={"Response": "Response", "Count": "Number of Respondents"},
)

fig.update_traces(textposition="outside")
fig.update_layout(width=800, height=500)
fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

col = "In the last month, have you been annoyed or troubled by a dog or dogs in your neighbourhood?"

# Count responses by district
counts = (
    df1_filtered.groupby("District")[col]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(index=["Pak Phli", "Ongkharak"])  # optional: ensure district order
)

# Optional: ensure columns ("Yes", "No", etc.) appear in consistent order
counts = counts.reindex(columns=["Yes", "No", "Don't know"], fill_value=0)

# Plot stacked bar
fig = go.Figure()

for response in counts.columns:
    fig.add_trace(go.Bar(
        x=counts.index,
        y=counts[response],
        name=response,
        text=counts[response],
        textposition="inside"
    ))

fig.update_layout(
    barmode="stack",
    title="Residents Troubled by Dogs in the Last Month (by District)",
    xaxis_title="District",
    yaxis_title="Number of Respondents",
    height=500,
    width=800,
    legend_title="Response",
    colorway=["#d16443", "#73c699", "#BDBDBD"]  # consistent colors
)

fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

col = "In the last month, have you been annoyed or troubled by a dog or dogs in your neighbourhood?"

# Count responses by district
counts = (
    df1_filtered.groupby("District")[col]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(index=["Pak Phli", "Ongkharak"])
)

# Ensure consistent response order
counts = counts.reindex(columns=["Yes", "No", "Don't know"], fill_value=0)

# Convert to percent
percent = counts.div(counts.sum(axis=1), axis=0) * 100

# Plot stacked bar
fig = go.Figure()

for response in percent.columns:
    fig.add_trace(go.Bar(
        x=percent.index,
        y=percent[response],
        name=response,
        text=percent[response].round(1).astype(str) + "%",
        textposition="inside"
    ))

fig.update_layout(
    barmode="stack",
    title="Residents Troubled by Dogs in the Last Month (by District)",
    xaxis_title="District",
    yaxis_title="Percent of Respondents (%)",
    height=500,
    width=900,
    legend_title="Response",
    colorway=["#d16443", "#73c699", "#BDBDBD"]
)

fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

col = "In the past 12 months, has anyone in your household been bitten by a dog?"

# Filter for valid responses only
df_bite = df1_filtered[df1_filtered[col].isin(["Yes", "No"])]

# Count by district and response
counts = (
    df_bite.groupby("District")[col]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(index=["Pak Phli", "Ongkharak"])  # keep order consistent
)

# --- Plot stacked bar ---
fig = go.Figure()
for response in ["Yes", "No"]:
    if response in counts.columns:
        fig.add_trace(go.Bar(
            x=counts.index,
            y=counts[response],
            name=response,
            text=counts[response],
            textposition="inside"
        ))

fig.update_layout(
    barmode="stack",
    title="Households Reporting Dog Bite Incidents (Past 12 Months)",
    xaxis_title="District",
    yaxis_title="Number of Households",
    height=500,
    width=800,
    legend_title="Response",
    colorway=["#d16443", "#73c699"]
)

fig.show()


In [ ]:
col = "In the past 12 months, has anyone in your household been bitten by a dog?"

# Filter valid responses
df_bite = df1_filtered[df1_filtered[col].isin(["Yes", "No"])]

# Count per district
counts_by_district = (
    df_bite.groupby("District")[col]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(index=["Pak Phli", "Ongkharak"])
)

# --- Make donut charts ---
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
)

color_map = {"No": "#d75b4f", "Yes": "#54C788"}

for i, district in enumerate(counts_by_district.index, start=1):
    counts = counts_by_district.loc[district]
    fig.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.6,  # <-- smaller hole (thicker donut)
            marker=dict(colors=[color_map[k] for k in counts.index]),
            textinfo="percent+label",
            name=district,
            hovertemplate="%{label}: %{value} households (%{percent})<extra></extra>"
        ),
        row=1, col=i
    )

# --- Add district annotations above donuts ---
annotations = []
for i, d in enumerate(counts_by_district.index):
    annotations.append(dict(
        text=f"<b>{d}</b>",
        x=0.185 if i == 0 else 0.835,
        y=0.5,
        showarrow=False,
        font=dict(size=16)
    ))

fig.update_layout(
    title="Dog Bite Incidents Reported by District (Past 12 Months)",
    height=600, width=1100,
    legend_title_text="Response",
    annotations=annotations
)

fig.show()


In [ ]:
col = "In the past 12 months, has anyone in your household been bitten by a dog?"

# Filter Yes/No only
df_bite = df1_filtered[df1_filtered[col].isin(["Yes", "No"])]

# Calculate proportions per district
bite_rate = (
    df_bite.groupby("District")[col]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(["Pak Phli", "Ongkharak"])
    * 100
)

print(bite_rate)


In [ ]:
# Merge bite data with soi dog counts
merge_cols = [
    "District",
    "Roughly how many dogs are roaming on your soi today?",
    "In the past 12 months, has anyone in your household been bitten by a dog?"
]

df_corr = df1_filtered[merge_cols].dropna()

# Encode Yes=1, No=0
df_corr["BiteBinary"] = (df_corr[col] == "Yes").astype(int)

# Correlation check
print(df_corr.groupby("District")[["Roughly how many dogs are roaming on your soi today?", "BiteBinary"]].corr().iloc[0::2, -1])


In [ ]:
import plotly.graph_objects as go

col_manage = "How should the roaming dog population in general be managed?"

# --- Aggregate counts by district ---
manage_counts = df1_filtered.groupby("District")[col_manage].value_counts().unstack(fill_value=0)

# --- Sort options by overall popularity ---
manage_counts = manage_counts[manage_counts.sum().sort_values(ascending=False).index]

# --- Create stacked bar chart ---
fig = go.Figure()
for method in manage_counts.columns:
    fig.add_trace(go.Bar(
        x=manage_counts.index,
        y=manage_counts[method],
        name=method
    ))

fig.update_layout(
    barmode="stack",
    title="Preferred Method for Managing Roaming Dog Population by District",
    xaxis_title="District",
    yaxis_title="Number of Responses",
    width=1100,
    height=600,
    legend_title_text="Management Method",
    template="plotly_white"
)

fig.show()


In [ ]:
print(df1_filtered.groupby("District")[col_manage].value_counts())

In [ ]:
import pandas as pd
import plotly.graph_objects as go

col_manage = "How should the roaming dog population in general be managed?"

# --- Group & count responses by district ---
manage_counts = df1_filtered.groupby("District")[col_manage].value_counts().reset_index(name="Count")

# --- Get top 5 answers for each district ---
top5_each = (
    manage_counts.groupby("District")
    .apply(lambda x: x.nlargest(5, "Count"))
    .reset_index(drop=True)
)

# --- Create figure ---
fig = go.Figure()

district_colors = {"Pak Phli": "#636efa", "Ongkharak": "#ef553b"}

for district in top5_each["District"].unique():
    subset = top5_each[top5_each["District"] == district]
    fig.add_trace(go.Bar(
        y=subset[col_manage],
        x=subset["Count"],
        name=district,
        orientation="h",
        marker_color=district_colors[district]
    ))

# --- Layout ---
fig.update_layout(
    barmode="group",
    title="Top 5 Preferred Methods for Managing Roaming Dogs (by District)",
    xaxis_title="Number of Respondents",
    yaxis_title="Management Approach",
    height=600,
    width=1000,
    legend_title_text="District",
    template="plotly_white"
)

fig.show()


In [ ]:
top5_each.value_counts

> Verbatim annoyance responses are intentionally excluded from this public notebook because free text and rare categories may create disclosure risk.
